In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from __future__ import print_function, division
import time
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import TensorFlow Keras instead of standalone Keras
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, concatenate, BatchNormalization
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import optimizers
from tensorflow.keras import backend as K  # Use TF's backend

warnings.filterwarnings("ignore")

In [ ]:
class AE_WGAN():
      def __init__(self):
          self.img_rows = 4
          self.img_cols = 6
          self.channels = 1
          self.img_shape = (self.img_rows, self.img_cols, self.channels)
          self.latent_dim = 32

          self.sum_time_generating = 0

          optimizer = optimizers.Adamax(.00009)

          #Build and compile the discriminator
          self.discriminator = self.build_discriminator()
          self.discriminator.compile(loss=[self.wasserstein_loss],
              optimizer=optimizer,
              metrics=['accuracy'])

          #Build the generator
          self.generator = self.build_generator()

          #Build the encoder
          self.encoder = self.build_encoder()

          #The part of the AE_WGAN that trains the discriminator and encoder
          self.discriminator.trainable = False

          #Generate image from sampled noise
          z = Input(shape=(self.latent_dim, ))
          img_ = self.generator(z)

          #Encode image
          img = Input(shape=self.img_shape)
          z_ = self.encoder(img)

          #Latent -> img is fake, and img -> latent is valid
          fake = self.discriminator([z, img_])
          valid = self.discriminator([z_, img])

          #Set up and compile the combined model
          #Trains generator to fool the discriminator
          self.AE_WGAN_generator = Model([z, img], [fake, valid])
          self.AE_WGAN_generator.compile(loss=[self.wasserstein_loss, self.wasserstein_loss],
              optimizer=optimizer)

      def wasserstein_loss(self, y_true, y_pred):
          return K.mean(y_true * y_pred)

      def build_encoder(self):
          model = Sequential()

          model.add(Flatten(input_shape=self.img_shape))
          model.add(Dense(20, activation='relu'))
          model.add(Dense(self.latent_dim))

          model.summary()

          img = Input(shape=self.img_shape)
          z = model(img)

          return Model(img, z)

      def build_generator(self):
          model = Sequential()

          model.add(Dense(40, input_dim=self.latent_dim, activation='relu'))
          model.add(Dense(80, activation='relu'))
          model.add(Dense(120, activation='relu'))

          model.add(Dense(np.prod(self.img_shape), activation='relu'))
          model.add(Reshape(self.img_shape))

          model.summary()

          z = Input(shape=(self.latent_dim,))
          gen_img = model(z)

          return Model(z, gen_img)

      def build_discriminator(self):
          model = Sequential()

          model.add(Dense(80, activation='relu', input_dim=self.latent_dim + np.prod(self.img_shape)))
          model.add(Dense(40, activation='relu'))
          model.add(Dense(1, activation='sigmoid'))

          model.summary()

          img = Input(shape=self.img_shape)
          z = Input(shape=(self.latent_dim,))

          # Flatten the image and concatenate it with z
          d_in = concatenate([z, Flatten()(img)])

          validity = model(d_in)

          return Model([z, img], validity)


      def train(self, attack_name, epochs, batch_size=32):

          print(self)
          epoch_list = []
          loss_D = []
          acc_D = []
          loss_G = []

          #Adversarial ground truths
          valid = np.ones((batch_size, 1))
          fake = np.zeros((batch_size, 1))

          for epoch in range(epochs):


              # ---------------------
              #  Train Discriminator
              # ---------------------

              #Sample noise and generate img
              z = np.random.normal(0, 1, size=(batch_size, self.latent_dim))
              imgs_ = self.generator.predict(z)

              #Select a random batch of images and encode
              idx = np.random.randint(0, reshaped_X_attack.shape[0], batch_size)
              imgs = reshaped_X_attack[idx]
              z_ = self.encoder.predict(imgs)

              #Train the discriminator (img -> z is valid, z -> img is fake)
              d_loss_real = self.discriminator.train_on_batch([z_, imgs], valid)
              d_loss_fake = self.discriminator.train_on_batch([z, imgs_], fake)
              d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

              # ---------------------
              #  Train Generator
              # ---------------------

              #Train the generator (z -> img is valid and img -> z is is invalid)
              g_loss = self.AE_WGAN_generator.train_on_batch([z, imgs], [valid, fake])

              #Plot the progress
              print ("%d [D loss: %f, acc: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100*d_loss[1], g_loss[0]))

              if epoch % 10 == 0:

                  loss_D.append(d_loss[0])
                  acc_D.append(100*d_loss[1])
                  loss_G.append(g_loss[0])
                  epoch_list.append(epoch)


              if epoch == 100:
                  print('\n')
                  print('Generating the attack data(',attack_name,')...\n')
                  generated_pics=self.generate_data()

          plt.figure(figsize=(10,7))
          plt.subplot(2, 1, 1)
          plt.title(attack_name)
          plt.plot(epoch_list, loss_D,'-o', label='Discriminator loss')
          plt.plot(epoch_list, loss_G,'-s', label='Generator loss')
          plt.ylabel('loss')
          plt.legend()

          plt.subplot(2, 1, 2)
          plt.plot(epoch_list, acc_D, '-s', label='Discriminator accuracy')
          plt.ylabel('accuracy')
          plt.xlabel('epochs')
          plt.legend()

          plt.savefig('./drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/images/AE_WGAN_NSL_KDD_%s.jpeg' % attack_name, bbox_inches='tight')
          plt.show()

          return generated_pics

      def generate_data(self):

          generated_pics = []

          a = int(len(reshaped_X_attack)+2000)

          for x in range(a):

              noise = np.random.normal(size=(32, self.latent_dim))

              gen_imgs = self.generator.predict(noise)

              generated_pics.append(gen_imgs[0])

          return generated_pics

In [ ]:
# Load NSL_KDD Dataset
# df=pd.read_csv('./drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/preprocessed_NSL_KDD.csv')
attack_column='label'

selected_attacks = ["back", "teardrop", "warezclient", "pod", "guess_passwd", "buffer_overflow", "warezmaster", "land", "imap", "rootkit", "loadmodule", "ftp_write", "multihop", "phf", "perl", "spy"]

# Create a directory to save the separated attacks
save_dir = "./drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/Generated_NSL_KDD/"
os.makedirs(save_dir, exist_ok=True)

for attack_name in selected_attacks:
    attack = pd.read_csv(f'./drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/Preprocessed_NSL_KDD/{attack_name}.csv')

    X_attack = attack.drop(columns=[attack_column], axis=1)

    X_attack = X_attack.astype(np.float32)
    reshaped_X_attack = np.asarray(X_attack).reshape(-1, 4, 6, 1)

    ae_wgan = AE_WGAN()
    generated_pics = ae_wgan.train(attack_name, epochs=101, batch_size=32)

    generated_attack = np.array(generated_pics).reshape(-1, 24)

    df_generated = pd.DataFrame(generated_attack, columns=X_attack.columns)

    for y in range(attack.shape[0]):
        df_generated['label'] = attack_name

    df_generated.to_csv(f"{save_dir}{attack_name}.csv", index=False)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 80)                  │           4,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 40)                  │           3,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              41 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,841 (30.63 KB)

 Trainable params: 7,841 (30.63 KB)

 Non-trainable params: 0 (0.00 B)

...

 Total params: 1,172 (4.58 KB)

 Trainable params: 1,172 (4.58 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
0 [D loss: 0.385824, acc: 83.59%] [G loss: 0.419927]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1 [D loss: 0.300233, acc: 80.73%] [G loss: 0.417631]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2 [D loss: 0.283405, acc: 79.84%] [G loss: 0.427822]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
3 [D loss: 0.276347, acc: 79.58%] [G loss: 0.432328]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
4 [D loss: 0.272249, acc: 78.65%] [G loss: 0.435170]


KeyboardInterrupt: 